# C5. The metaplectic operator $\mathcal U(g_\alpha)$ at intermediate angles

Companion notebook to the bachelor's thesis *El formalismo ODM en sistemas híbridos clásico-cuánticos* (Santiago Puyol Miano, Universidad de Zaragoza, 2026).

On the quantum side, the symplectic transformation $g_\alpha$ of C2 is implemented by the unitary operator $\mathcal U(g_\alpha)=e^{-i\alpha\hat K}$, with $\hat K=\tfrac12(\hat p^2+\hat\lambda_p^2)$ and $[\hat p,\hat\lambda_p]=i$. It satisfies the equivariance

$$\mathcal U(g_\alpha)\,\hat p\,\mathcal U(g_\alpha)^{-1}=\cos\alpha\,\hat p-\sin\alpha\,\hat\lambda_p,\qquad
\mathcal U(g_\alpha)\,\hat\lambda_p\,\mathcal U(g_\alpha)^{-1}=\sin\alpha\,\hat p+\cos\alpha\,\hat\lambda_p,$$

which matches the composition of $p$ and $\lambda_p$ with $g_\alpha^{-1}$, and its integral kernel is the Mehler kernel. This notebook checks the classical formulas symbolically with sympy, and the equivariance, unitarity and group law numerically with numpy, at intermediate values of $\alpha$ away from the endpoints $0$ and $\pi/2$.

**Kernel:** Python 3 with sympy and numpy.

## Notation

- Coordinates $(x,p,\lambda_x,\lambda_p)$ on $\Xi=T^*\mathbb R^2$, written `x p lx lp` in the code; $a$ is the angle $\alpha$.
- $g_\alpha=e^{\alpha\xi}$ rotates the pair $(p,\lambda_p)$ and fixes $(x,\lambda_x)$. The matrix $J$ encodes $\{x,\lambda_x\}=\{p,\lambda_p\}=1$.
- κ-identification: $\hat x_q=\hat x-\tfrac{\hbar\kappa}{2}\hat\lambda_p$ and $\hat p_q=\hat p+\tfrac{\hbar\kappa}{2}\hat\lambda_x$, with $\hat\lambda_x=-i\partial_x$ and $\hat\lambda_p=-i\partial_p$.

In [1]:
import sympy as sp
import numpy as np

a = sp.symbols('alpha', real=True)
hbar, kappa = sp.symbols('hbar kappa', positive=True)

## 1. $g_\alpha$ is symplectic, and its action on coordinates

The cell computes $g_\alpha=e^{\alpha\xi}$ and $g_\alpha^{-1}$, checks $g_\alpha^\top Jg_\alpha=J$, and checks

$$p\circ g_\alpha^{-1}=\cos\alpha\,p-\sin\alpha\,\lambda_p,\qquad \lambda_p\circ g_\alpha^{-1}=\sin\alpha\,p+\cos\alpha\,\lambda_p,\qquad p\circ g_{\pi/2}^{-1}=-\lambda_p.$$

The Hamiltonian vector field of $K=\tfrac12(p^2+\lambda_p^2)$ on the $(p,\lambda_p)$ block is $X_K=\lambda_p\,\partial_p-p\,\partial_{\lambda_p}$, and its flow is the rotation block of $g_\alpha$.

In [2]:
xi = sp.Matrix([[0,0,0,0],[0,0,0,1],[0,0,0,0],[0,-1,0,0]])
J  = sp.Matrix([[0,0,1,0],[0,0,0,1],[-1,0,0,0],[0,-1,0,0]])  # {x,lx}=1,{p,lp}=1
g  = sp.simplify(sp.exp(a*xi))
print('g_alpha =')
sp.pprint(g)
print('g^T J g - J =', sp.simplify(g.T*J*g - J))

ginv = sp.simplify(g.inv())
print('g^{-1} =')
sp.pprint(ginv)
# z = (x, p, lx, lp); the entries of g^{-1} z are the coordinate functions composed with g^{-1}
z = sp.Matrix(sp.symbols('x p lx lp', real=True))
zc = ginv*z
print('p o g^{-1}  =', sp.simplify(zc[1]))   # expect cos a * p - sin a * lp
print('lp o g^{-1} =', sp.simplify(zc[3]))   # expect sin a * p + cos a * lp
print('p o g^{-1} at pi/2 =', sp.simplify(zc[1].subs(a, sp.pi/2)))  # expect -lp

assert sp.simplify(g.T*J*g - J) == sp.zeros(4, 4), "g_alpha not symplectic"
assert sp.simplify(zc[1] - (sp.cos(a)*z[1] - sp.sin(a)*z[3])) == 0, "p o g^-1 wrong"
assert sp.simplify(zc[3] - (sp.sin(a)*z[1] + sp.cos(a)*z[3])) == 0, "lp o g^-1 wrong"
assert sp.simplify(zc[1].subs(a, sp.pi/2) + z[3]) == 0, "p o g^-1 at pi/2 != -lp"
print('OK: g_alpha is symplectic, and p o g^-1, lp o g^-1 are as expected, with p o g^-1 = -lp at pi/2')

g_alpha =
⎡1     0     0    0   ⎤
⎢                     ⎥
⎢0  cos(α)   0  sin(α)⎥
⎢                     ⎥
⎢0     0     1    0   ⎥
⎢                     ⎥
⎣0  -sin(α)  0  cos(α)⎦
g^T J g - J = Matrix([[0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0]])
g^{-1} =
⎡1    0     0     0   ⎤
⎢                     ⎥
⎢0  cos(α)  0  -sin(α)⎥
⎢                     ⎥
⎢0    0     1     0   ⎥
⎢                     ⎥
⎣0  sin(α)  0  cos(α) ⎦
p o g^{-1}  = -lp*sin(alpha) + p*cos(alpha)
lp o g^{-1} = lp*cos(alpha) + p*sin(alpha)
p o g^{-1} at pi/2 = -lp
OK: g_alpha is symplectic, and p o g^-1, lp o g^-1 are as expected, with p o g^-1 = -lp at pi/2


## 2. The Heisenberg equations

The operators $A(\alpha)=\mathcal U_\alpha\hat p\,\mathcal U_\alpha^{-1}$ and $B(\alpha)=\mathcal U_\alpha\hat\lambda_p\,\mathcal U_\alpha^{-1}$ satisfy $A'=-B$, $B'=A$ with $A(0)=\hat p$ and $B(0)=\hat\lambda_p$. The cell checks that $\cos\alpha$ and $\sin\alpha$ solve this system, which gives the equivariance formulas above.

In [3]:
print('d(cos)/da + sin =', sp.simplify(sp.diff(sp.cos(a),a)+sp.sin(a)))
print('d(sin)/da - cos =', sp.simplify(sp.diff(sp.sin(a),a)-sp.cos(a)))

assert sp.simplify(sp.diff(sp.cos(a), a) + sp.sin(a)) == 0
assert sp.simplify(sp.diff(sp.sin(a), a) - sp.cos(a)) == 0
print("OK: A = cos, B = sin solve A' = -B, B' = A")

d(cos)/da + sin = 0
d(sin)/da - cos = 0
OK: A = cos, B = sin solve A' = -B, B' = A


## 3. The Mehler kernel at $\alpha=\pi/2$

For $0<\alpha<\pi$,

$$K_\alpha(p,p')=\frac{1}{\sqrt{2\pi i\sin\alpha}}\;e^{\frac{i}{2\sin\alpha}\left[(p^2+p'^2)\cos\alpha-2pp'\right]}.$$

At $\alpha=\pi/2$ it becomes $(2\pi)^{-1/2}e^{-i\pi/4}e^{-ipp'}$, the Fourier transform in $p$ up to the phase $e^{-i\pi/4}$. The cell prints the simplified kernel. Sympy writes the constant as $-\sqrt2\,i^{3/2}/(2\sqrt\pi)$, which equals $e^{-i\pi/4}/\sqrt{2\pi}$.

In [4]:
p, pp = sp.symbols('p pp', real=True)
K = 1/sp.sqrt(2*sp.pi*sp.I*sp.sin(a)) * sp.exp(sp.I/(2*sp.sin(a))*((p**2+pp**2)*sp.cos(a) - 2*p*pp))
print('K_{pi/2} =', sp.simplify(K.subs(a, sp.pi/2)))

K_{pi/2} = -sqrt(2)*I**(3/2)*exp(-I*p*pp)/(2*sqrt(pi))


## 4. $[\hat x_q,\hat p_q]=i\hbar\kappa$ in the vertical representation

On functions $\psi(x,p)$ the κ-identification acts as $\hat x_q=x+\tfrac{i\hbar\kappa}{2}\partial_p$ and $\hat p_q=p-\tfrac{i\hbar\kappa}{2}\partial_x$.

In [5]:
x, pv = sp.symbols('x p', real=True)
psi = sp.Function('psi')(x, pv)
xq  = lambda f: x*f + sp.I*hbar*kappa/2*sp.diff(f, pv)
pq  = lambda f: pv*f - sp.I*hbar*kappa/2*sp.diff(f, x)
comm = sp.simplify(xq(pq(psi)) - pq(xq(psi)))
print('[x_q,p_q] psi =', comm, '  (expect i*hbar*kappa*psi)')

assert sp.simplify(comm - sp.I*hbar*kappa*psi) == 0, "[x_q,p_q] != i hbar kappa"
print('OK: [x_q, p_q] psi = i*hbar*kappa*psi')

[x_q,p_q] psi = I*hbar*kappa*psi(x, p)   (expect i*hbar*kappa*psi)
OK: [x_q, p_q] psi = i*hbar*kappa*psi


## 5. Equivariance, unitarity and group law at intermediate $\alpha$ (numerical)

In the Hermite (Fock) basis, $\hat p=(a+a^\dagger)/\sqrt2$ and $\hat\lambda_p=-i(a-a^\dagger)/\sqrt2$, so that $[\hat p,\hat\lambda_p]=i$, and $\hat K=N+\tfrac12$. Then $\mathcal U_\alpha=e^{-i\alpha(N+1/2)}$ is diagonal. The basis is truncated at dimension 60, and the comparisons use an interior block away from the truncation edge. The cell checks the commutator, unitarity and both equivariance formulas at $\alpha=0.3,\ \pi/4,\ 1,\ 0.4\pi$, and the group law $\mathcal U_a\mathcal U_b=\mathcal U_{a+b}$, all with tolerance $10^{-9}$.

In [6]:
def check_equivariance(alpha, dim=60):
    n = np.arange(dim)
    ad = np.diag(np.sqrt(np.arange(1, dim)), -1)   # a^dagger (raising)
    aa = ad.conj().T                                # a (lowering)
    P  = (aa + ad) / np.sqrt(2)
    Lp = -1j * (aa - ad) / np.sqrt(2)
    # bracket [P,Lp] = i  (interior block; edge truncation ignored)
    comm = (P @ Lp - Lp @ P)
    U  = np.diag(np.exp(-1j * alpha * (n + 0.5)))
    Ud = U.conj().T
    lhs_p  = U @ P  @ Ud
    rhs_p  = np.cos(alpha) * P - np.sin(alpha) * Lp
    lhs_lp = U @ Lp @ Ud
    rhs_lp = np.sin(alpha) * P + np.cos(alpha) * Lp
    # compare on an interior block (away from the truncation edge)
    s = slice(0, dim - 6)
    e_p  = np.max(np.abs((lhs_p  - rhs_p )[s, s]))
    e_lp = np.max(np.abs((lhs_lp - rhs_lp)[s, s]))
    e_comm = np.max(np.abs((comm - 1j*np.eye(dim))[s, s]))
    e_unit = np.max(np.abs((U @ Ud - np.eye(dim))))
    return e_comm, e_unit, e_p, e_lp

print("Equivariance  U p U^-1 = cos a p - sin a lp  (intermediate alpha):")
for alpha in [0.3, 0.7853981633974483, 1.0, 1.2566370614359172]:  # includes pi/4 and 0.4*pi
    e_comm, e_unit, e_p, e_lp = check_equivariance(alpha)
    print(f"  alpha={alpha:.4f}: |[p,lp]-i|={e_comm:.2e}  |UU*-1|={e_unit:.2e}"
          f"  err_p={e_p:.2e}  err_lp={e_lp:.2e}")

# group law U_a U_b = U_{a+b}
dim=60; n=np.arange(dim)
Ua=np.diag(np.exp(-1j*0.3*(n+0.5))); Ub=np.diag(np.exp(-1j*0.7*(n+0.5)))
Uab=np.diag(np.exp(-1j*1.0*(n+0.5)))
group_err = np.max(np.abs(Ua@Ub-Uab))
print(f"  group law |U_a U_b - U_(a+b)| = {group_err:.2e}")

TOL = 1e-9
for _alpha in [0.3, 0.7853981633974483, 1.0, 1.2566370614359172]:
    _e = check_equivariance(_alpha)
    assert max(_e) < TOL, f"equivariance/unitarity failed at alpha={_alpha}: {_e}"
assert group_err < TOL, "one-parameter group law failed"
print("OK: commutator, unitarity and equivariance within 1e-9 at every tested alpha; group law within 1e-9")

Equivariance  U p U^-1 = cos a p - sin a lp  (intermediate alpha):
  alpha=0.3000: |[p,lp]-i|=1.71e-14  |UU*-1|=2.22e-16  err_p=5.56e-15  err_lp=5.56e-15
  alpha=0.7854: |[p,lp]-i|=1.71e-14  |UU*-1|=2.24e-16  err_p=2.29e-14  err_lp=2.29e-14
  alpha=1.0000: |[p,lp]-i|=1.71e-14  |UU*-1|=2.22e-16  err_p=9.93e-16  err_lp=9.93e-16
  alpha=1.2566: |[p,lp]-i|=1.71e-14  |UU*-1|=2.22e-16  err_p=3.67e-14  err_lp=3.67e-14
  group law |U_a U_b - U_(a+b)| = 7.16e-15
OK: commutator, unitarity and equivariance within 1e-9 at every tested alpha; group law within 1e-9


In [7]:
# Every assert above has passed if this cell runs.
print('C5: all checks passed.')

C5: all checks passed.
